# PPL Interactive Interface
This notebook allows you to explore the Tennessee Eastman Process dataset, define model inputs/outputs, train the pipeline, and visualize results.


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.pipeline import PPLPipeline
from src.preprocessing import apply_savitzky_golay

%load_ext autoreload
%autoreload 2
%matplotlib inline


/Users/russellbell/Documents/Personal/Coding/Predictive Performance Labs (PPL)/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
data_path = '../data/python_data_1year.csv'
if not os.path.exists(data_path):
    print("Data file not found! Please run src/download_data.py first.")
else:
    df = pd.read_csv(data_path, sep=';')
    # Handle Timestamp
    if 'timestamp' not in df.columns:
        # Assume first column is timestamp
        first_col = df.columns[0]
        df.rename(columns={first_col: 'timestamp'}, inplace=True)
        print(f"Renamed first column '{first_col}' to 'timestamp'.")
    
    # Ensure datetime if possible
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    except Exception as e:
        print(f"Could not convert timestamp to datetime: {e}")
        
    print(f"Loaded data: {df.shape}")
    print(df.describe())
    
    # Interactive Data Distribution Explorer
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    summary_output = widgets.Output()
    
    # Dropdown for variable selection
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    var_dropdown = widgets.Dropdown(
        options=numeric_cols,
        value=numeric_cols[0] if numeric_cols else None,
        description='Variable:',
        layout=widgets.Layout(width='300px')
    )
    
    # Use make_subplots to properly initialize axes
    fig_summary = go.FigureWidget(make_subplots(rows=1, cols=2, subplot_titles=("Distribution", "Statistics")))
    
    fig_summary.update_layout(
        title_text='Distribution Explorer',
        height=400,
        showlegend=False,
        template='plotly_white'
    )
    
    def update_summary(change):
        col = var_dropdown.value
        if not col or col not in df.columns:
            return
            
        data = df[col].dropna()
        mean_val = data.mean()
        std_val = data.std()
        
        with fig_summary.batch_update():
            trace1 = go.Histogram(
                x=data, 
                name='Dist', 
                marker_color='royalblue', 
                opacity=0.7
            )
            
            trace2 = go.Box(
                y=data, 
                name='Stats', 
                marker_color='lightblue',
                boxpoints='outliers'
            )
            
            # Replace data
            fig_summary.data = []
            fig_summary.add_trace(trace1, row=1, col=1)
            fig_summary.add_trace(trace2, row=1, col=2)
            
            fig_summary.update_layout(
                title_text=f"Distribution: {col} (Mean: {mean_val:.2f}, Std: {std_val:.2f})"
            )
            fig_summary.update_xaxes(title_text="Value", row=1, col=1)
            fig_summary.update_yaxes(title_text="Count", row=1, col=1)
            fig_summary.update_yaxes(title_text="Value", row=1, col=2)

    var_dropdown.observe(update_summary, names='value')
    
    # Initial Call
    try:
        update_summary(None)
    except Exception as e:
        with summary_output:
            print(f"Error updating plot: {e}")
    
    display(widgets.VBox([
        widgets.Label("Select a variable to view its distribution and summary statistics:"),
        var_dropdown,
        fig_summary,
        summary_output
    ]))

Loaded data: (175200, 54)
                 timestamp       XMEAS(1)       XMEAS(2)       XMEAS(3)  \
count               175200  175200.000000  175200.000000  175200.000000   
mean   1970-07-02 11:58:30       0.265182    3664.039842    4507.001212   
min    1970-01-01 00:00:00      -0.003547    3411.558519    3920.192284   
25%    1970-04-02 05:59:15       0.223320    3639.409615    4474.434282   
50%    1970-07-02 11:58:30       0.251870    3664.209011    4507.916514   
75%    1970-10-01 17:57:45       0.281207    3688.735150    4541.186582   
max    1970-12-31 23:57:00       1.007899    3862.611783    5095.277955   
std                    NaN       0.112623      37.868760      75.487832   

            XMEAS(4)       XMEAS(5)       XMEAS(6)       XMEAS(7)  \
count  175200.000000  175200.000000  175200.000000  175200.000000   
mean        9.331623      26.902240      42.338699    2706.222178   
min         7.568992      25.529484      40.693448    2490.704826   
25%         9.265495  

In [4]:
# Unified Data Exploration & Filtering (Subplots)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import widgets
import numpy as np
import pandas as pd

# Variable Selection
variables = [c for c in df.columns if c != 'timestamp']
style = {'description_width': 'initial'}

# --- 1. Controls ---
# Plot Controls
plot_selector = widgets.SelectMultiple(
    options=variables,
    value=[variables[0]],
    description='Variables:',
    layout=widgets.Layout(width='40%', height='150px'),
    style=style
)

range_slider = widgets.IntRangeSlider(
    value=[0, 1000],
    min=0,
    max=len(df),
    step=100,
    description='Range:',
    layout=widgets.Layout(width='50%')
)
normalize_check = widgets.Checkbox(value=False, description='Normalize', indent=False)

# Bad Data Rule Controls
rule_var = widgets.Dropdown(options=variables, description='Bad Var:')
rule_cond = widgets.Dropdown(options=['<', '>', '='], description='Cond:', layout=widgets.Layout(width='80px'))
rule_val = widgets.FloatText(value=0.0, description='Val:', layout=widgets.Layout(width='100px'))
add_rule_btn = widgets.Button(description='Add Rule', button_style='info', icon='plus')
reset_rules_btn = widgets.Button(description='Reset Rules', button_style='warning', icon='refresh')
apply_rules_btn = widgets.Button(description='Apply Filter', button_style='success', icon='check')

rules_list = []
status_output = widgets.Output()
plot_container = widgets.VBox()

# --- 2. Logic ---
def get_filtered_mask(df_subset):
    if not rules_list:
        return pd.Series(True, index=df_subset.index)
        
    bad_mask = pd.Series(False, index=df_subset.index)
    for r in rules_list:
        v, c, val = r['var'], r['cond'], r['val']
        if v not in df_subset.columns: continue
        
        if c == '<': sub_bad = df_subset[v] < val
        elif c == '>': sub_bad = df_subset[v] > val
        elif c == '=': sub_bad = df_subset[v] == val
        
        bad_mask = bad_mask | sub_bad
        
    return ~bad_mask

def update_plot(change=None):
    with status_output:
        plot_container.children = () # Clear current plot
        
        # 1. Get Data Range
        start, end = range_slider.value
        if start >= end: return
        
        sub_df = df.iloc[start:end].copy()
        mask = get_filtered_mask(sub_df)
        
        selected_vars = list(plot_selector.value)
        if not selected_vars:
            return
        
        # 2. Create Subplots
        n_rows = len(selected_vars)
        fig = go.FigureWidget(make_subplots(
            rows=n_rows, cols=1, 
            shared_xaxes=True, 
            vertical_spacing=0.05,
            subplot_titles=selected_vars
        ))
        
        fig.update_layout(
            height=300 * n_rows,
            title=f'Data: {start}-{end}',
            template='plotly_white',
            showlegend=False,
             margin=dict(t=50, b=20, l=40, r=40)
        )

        # 3. Add Traces
        for i, col in enumerate(selected_vars):
            y_vals = sub_df[col]
            # Normalize if checked
            if normalize_check.value:
                min_v, max_v = df[col].min(), df[col].max()
                denom = max_v - min_v
                if denom != 0:
                     y_vals = (y_vals - min_v) / denom
            
            fig.add_trace(go.Scatter(
                x=sub_df['timestamp'], 
                y=y_vals,
                name=col,
                mode='lines'
            ), row=i+1, col=1)
            
        # 4. Highlight Bad Data
        if len(rules_list) > 0:
             bad_indices = sub_df.index[~mask]
             if len(bad_indices) > 0:
                 try:
                     is_bad = (~mask).values.astype(int)
                     padded = np.concatenate(([0], is_bad, [0]))
                     diffs = np.diff(padded)
                     starts = np.where(diffs == 1)[0]
                     ends = np.where(diffs == -1)[0]
                     
                     timestamps = sub_df['timestamp'].values
                     
                     # Collect shapes for ALL rows
                     for i in range(min(len(starts), 50)):
                         s, e = starts[i], ends[i]
                         t_start = timestamps[s]
                         t_end = timestamps[e-1] if e-1 < len(timestamps) else timestamps[-1]
                         
                         # Convert to str for JSON safety
                         t_start_str = str(t_start)
                         t_end_str = str(t_end)
                         
                         # Add vrect to ALL rows
                         fig.add_vrect(
                             x0=t_start_str, x1=t_end_str,
                             fillcolor="red", opacity=0.1, layer="below", line_width=0,
                             row="all", col=1
                         )
                 except Exception as e:
                     print(f"Error highlighting: {e}")
        
        plot_container.children = (fig,)

def on_add_rule(b):
    rules_list.append({'var': rule_var.value, 'cond': rule_cond.value, 'val': rule_val.value})
    with status_output: clear_output()
    update_plot()

def on_reset(b):
    global rules_list
    rules_list = []
    if os.path.exists('../data/temp_filtered_data.csv'):
        os.remove('../data/temp_filtered_data.csv')
    with status_output: clear_output(); print("Rules reset.")
    update_plot()

def on_apply(b):
    with status_output:
        print("Applying rules to FULL dataset...")
        mask = get_filtered_mask(df)
        filtered_df = df[mask]
        save_path = '../data/temp_filtered_data.csv'
        filtered_df.to_csv(save_path, sep=';', index=False)
        print(f"Saved filtered data: {len(filtered_df)} rows (removed {len(df)-len(filtered_df)}).")

# Bindings
plot_selector.observe(lambda x: update_plot(None), names='value')
range_slider.observe(lambda x: update_plot(None), names='value')
normalize_check.observe(lambda x: update_plot(None), names='value')
add_rule_btn.on_click(on_add_rule)
reset_rules_btn.on_click(on_reset)
apply_rules_btn.on_click(on_apply)

# Layout
controls = widgets.VBox([
    widgets.HBox([widgets.Label("Select Variables:"), plot_selector]),
    widgets.HBox([range_slider, normalize_check]),
    widgets.HTML("<hr>"),
    widgets.HBox([widgets.Label("Bad Data Rule:"), rule_var, rule_cond, rule_val, add_rule_btn]),
    widgets.HBox([reset_rules_btn, apply_rules_btn]),
])

display(controls, status_output, plot_container)
update_plot(None)

Output()

VBox()

In [5]:
# Model Configuration
print("Configure Model Inputs (MVs/DVs) and Outputs (CVs)")

# Identify likely MVs, DVs, CVs based on naming convention
mvs = [c for c in variables if 'XMV' in c]
dvs = [c for c in variables if 'fault' in c or 'IDV' in c] # Adjust if DVs have specific names
cvs = [c for c in variables if 'XMEAS' in c]

# If no specific naming, just list all
if not mvs: mvs = variables
if not cvs: cvs = variables

input_selector = widgets.SelectMultiple(
    options=variables,
    value=mvs[:5] if mvs else variables[:5],
    description='Inputs (MVs/DVs):',
    layout=widgets.Layout(width='45%', height='200px'),
    style=style
)

output_selector = widgets.SelectMultiple(
    options=variables,
    value=cvs[:5] if cvs else variables[:5],
    description='Outputs (CVs):',
    layout=widgets.Layout(width='45%', height='200px'),
    style=style
)

display(widgets.HBox([input_selector, output_selector]))


Configure Model Inputs (MVs/DVs) and Outputs (CVs)


In [9]:
from datetime import datetime
# Training Controls
train_button = widgets.Button(description="Train Model", button_style='success')
# [NEW] Experiment Name Input
exp_name_input = widgets.Text(value=f'experiment_{datetime.now().strftime("%Y%m%d_%H%M%S")}', description='Exp Name:')
train_output = widgets.Output()

def on_train_click(b):
    global pipeline
    with train_output:
        clear_output()
        print("Initializing Pipeline...")
        
        input_cols = list(input_selector.value)
        output_cols = list(output_selector.value)
        run_name = exp_name_input.value
        
        if not input_cols or not output_cols:
            print("Error: Please select at least one input and one output.")
            return
            
        print(f"Inputs: {len(input_cols)} variables")
        print(f"Outputs: {len(output_cols)} variables")
        print(f"Run Name: {run_name}")
        
        # Check for filtered data
        current_data_path = data_path
        if os.path.exists('../data/temp_filtered_data.csv'):
            print("Using filtered data from Bad Data Rules.")
            current_data_path = '../data/temp_filtered_data.csv'
        else:
            print("Using original full dataset.")
        
        # Initialize pipeline with correct data
        pipeline = PPLPipeline(current_data_path, output_dir='../output')
        
        # Run pipeline
        # Using small sample size for interactive demo
        print("Running pipeline...")
        # [NEW] Pass run_name
        results = pipeline.run(input_cols=input_cols, output_cols=output_cols, sample_size=2000, epochs=1000, run_name=run_name)
        history = results['history']
        mse = results['mse']
        # Attach pipeline to button for access in other cells
        train_button.pipeline = pipeline
        
        print(f"Training Complete. Final MSE: {mse:.4f}")
        
        plt.close('all') # Clear any previous plots
        
        # Plot Loss
        plt.figure(figsize=(10, 4))
        plt.plot(history.history['loss'], label='Train Loss')
        if 'val_loss' in history.history:
            plt.plot(history.history['val_loss'], label='Val Loss')
        plt.title('Model Training History')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.show()
        
        # Show generated artifacts
        print("Displaying generated visualizations...")
        try:
            from IPython.display import Image, display
            # Construct paths based on run_name
            base_path = f'../output/{run_name}'
            display(Image(filename=f'{base_path}/error_dist.png'))
            display(Image(filename=f'{base_path}/lime_violin.png'))
            print(f"Config and model saved to: {base_path}/")
        except Exception as e:
            print(f"Could not display images: {e}")

train_button.on_click(on_train_click)
display(widgets.HBox([train_button, exp_name_input]), train_output)

Output()

## Random Scenario Exploration
Visualize model performance on random samples from the test set.

In [10]:
random_viz_button = widgets.Button(description="Pick Random Scenario", button_style='info')
random_viz_output = widgets.Output()

def on_random_viz_click(b):
    with random_viz_output:
        clear_output(wait=True)
        # Retrieve pipeline from train button
        pipeline = getattr(train_button, 'pipeline', None)
        if pipeline is None or not hasattr(pipeline, 'model'):
            print("Error: Model not trained yet. Please train the model first.")
            return
            
        # Pick random index
        idx = np.random.randint(0, len(pipeline.X_images))
        print(f"Visualizing Sample Index: {idx}")
        
        # Get Data
        if not hasattr(pipeline, 'X_windows'):
            print("Error: pipeline.X_windows not found. Please re-run training.")
            return
            
        input_data = pipeline.X_windows[idx] # Shape: (window_size, n_inputs)
        true_output = pipeline.y_ts[idx]     # Shape: (window_size, n_outputs)
        
        # Predict
        input_img = pipeline.X_images[idx]
        pred_output = pipeline.model.predict(input_img[np.newaxis, ...], verbose=0)[0]
        
        # Setup Vars
        input_cols = list(input_selector.value)
        output_cols = list(output_selector.value)
        time_steps = np.arange(len(true_output))
        
        # Layout calculations
        n_in = len(input_cols)
        n_out = len(output_cols)
        cols = 4
        rows_in = (n_in - 1) // cols + 1
        rows_out = (n_out - 1) // cols + 1
        total_rows = rows_in + rows_out
        
        plt.figure(figsize=(15, 3 * total_rows))
        
        # --- 1. Plot Inputs ---
        for i, col in enumerate(input_cols):
            plt.subplot(total_rows, cols, i + 1)
            if i < input_data.shape[1]:
                 plt.plot(time_steps, input_data[:, i], 'g-', label='Input')
            plt.title(col, fontsize=10)
            plt.grid(True, alpha=0.3)
            if i == 0: plt.text(0.05, 1.1, "INPUTS", transform=plt.gca().transAxes, fontsize=12, fontweight='bold', color='green')
            
        # --- 2. Plot Outputs ---
        # Start index for outputs: (rows_in * cols) + 1 to force new row
        start_idx = rows_in * cols + 1
        for i, col in enumerate(output_cols):
            ax = plt.subplot(total_rows, cols, start_idx + i)
            if i < true_output.shape[1]:
                plt.plot(time_steps, true_output[:, i], 'b-', label='Actual')
                plt.plot(time_steps, pred_output[:, i], 'r--', label='Predicted')
            plt.title(col, fontsize=10)
            plt.grid(True, alpha=0.3)
            if i == 0: 
                plt.legend(prop={'size': 8})
                plt.text(0.05, 1.1, "OUTPUTS", transform=plt.gca().transAxes, fontsize=12, fontweight='bold', color='blue')
            
        plt.suptitle(f"Scenario {idx}", fontsize=14)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

random_viz_button.on_click(on_random_viz_click)
display(random_viz_button, random_viz_output)

Button(button_style='info', description='Pick Random Scenario', style=ButtonStyle())

Output()

## Interactive Error Analysis
Select a range of errors in the histogram below to see the distribution of input features for those specific samples.

In [11]:
import plotly.graph_objects as go
from ipywidgets import widgets
import pandas as pd
import numpy as np

# 1. Prepare Data
if 'pipeline' not in globals():
    pipeline = getattr(train_button, 'pipeline', None)

if pipeline is None or not hasattr(pipeline, 'model'):
    print("Please train the model first.")
else:
    # --- Data Calculation ---
    print("Calculating predictions for all samples...")
    all_preds = pipeline.model.predict(pipeline.X_images, verbose=0)
    
    residuals = pipeline.y_ts - all_preds
    residuals_flat = residuals.flatten()
    
    n_samples, n_timestamps, n_features = residuals.shape
    sample_indices_flat = np.repeat(np.arange(n_samples), n_timestamps * n_features)
    
    input_features_mean = np.mean(pipeline.X_windows, axis=1)
    input_col_names = list(input_selector.value)
    
    lime_df = None
    if hasattr(pipeline, 'lime_importances'):
        lime_df = pd.DataFrame(pipeline.lime_importances)
        print(f"LIME data available for {len(lime_df)} samples.")
    else:
        print("LIME data not found. Please retrain the model.")
    
    # --- Figure Creation ---
    
    # 1. Error Histogram
    trace_error_all = go.Histogram(x=residuals_flat, name='All Residuals', marker_color='lightgrey', opacity=0.5)
    trace_error_sel = go.Histogram(x=[], name='Selected', marker_color='blue', opacity=1.0)
    
    error_fig = go.FigureWidget(
        data=[trace_error_all, trace_error_sel],
        layout=go.Layout(
            title='Prediction Error (Raw Residuals)',
            xaxis_title='Residual (Actual - Predicted)',
            yaxis_title='Count',
            dragmode='select',
            height=300,
            barmode='overlay',
            showlegend=True
        )
    )
    
    # 2. Input Histograms
    n_features_to_plot = min(4, len(input_col_names))
    input_figs = []
    
    for i in range(n_features_to_plot):
        col_name = input_col_names[i]
        feature_data = input_features_mean[:, i]
        
        t_all = go.Histogram(x=feature_data, name='All', marker_color='lightgrey', opacity=0.5, nbinsx=50)
        t_sel = go.Histogram(x=[], name='Selected', marker_color='red', opacity=0.8, nbinsx=50)
        
        fig = go.FigureWidget(
            data=[t_all, t_sel],
            layout=go.Layout(
                title=f'Input: {col_name}',
                height=250,
                showlegend=False,
                barmode='overlay',
                dragmode='select',
                margin=dict(l=20, r=20, t=40, b=20)
            )
        )
        input_figs.append(fig)
        
    # 3. LIME Violin Plot
    lime_fig = None
    if lime_df is not None:
        lime_traces = []
        for col in lime_df.columns:
            lime_traces.append(go.Violin(
                y=lime_df[col], 
                name=col, 
                line_color='lightgrey', 
                points='all', 
                side='negative',
                showlegend=False
            ))
            lime_traces.append(go.Violin(
                y=[], 
                name=col, 
                line_color='green', 
                points=False, 
                side='positive',
                showlegend=False
            ))
            
        lime_fig = go.FigureWidget(
            data=lime_traces,
            layout=go.Layout(
                title='LIME Feature Importance',
                yaxis_title='Importance',
                height=400,
                dragmode='select',
                violingap=0, violinmode='overlay'
            )
        )
    
    # --- Interaction Logic ---
    
    def update_all_plots(selected_sample_indices):
        # Update Error Plot
        with error_fig.batch_update():
            if len(selected_sample_indices) > 0:
                mask = np.isin(sample_indices_flat, selected_sample_indices)
                error_fig.data[1].x = residuals_flat[mask]
            else:
                error_fig.data[1].x = []
            
        # Update Input Plots
        for i, fig in enumerate(input_figs):
            with fig.batch_update():
                if len(selected_sample_indices) > 0:
                    fig.data[1].x = input_features_mean[selected_sample_indices, i]
                else:
                    fig.data[1].x = []
                
        # Update LIME Plot
        if lime_fig is not None:
            lime_indices = [idx for idx in selected_sample_indices if idx < len(lime_df)]
            with lime_fig.batch_update():
                if len(lime_indices) > 0:
                    subset = lime_df.iloc[lime_indices]
                    for i, col in enumerate(lime_df.columns):
                        lime_fig.data[2*i + 1].y = subset[col]
                else:
                    for i, col in enumerate(lime_df.columns):
                        lime_fig.data[2*i + 1].y = []

    def error_selection_fn(trace, points, selector):
        if not selector or not selector.xrange:
            update_all_plots([])
            return
        min_err, max_err = selector.xrange
        flat_indices = np.where((residuals_flat >= min_err) & (residuals_flat <= max_err))[0]
        if len(flat_indices) > 0:
            selected_sample_indices = np.unique(sample_indices_flat[flat_indices])
        else:
            selected_sample_indices = []
        update_all_plots(selected_sample_indices)

    def input_selection_fn_factory(feature_idx):
        def input_selection_fn(trace, points, selector):
            if not selector or not selector.xrange:
                update_all_plots([])
                return
            min_val, max_val = selector.xrange
            # Find samples where this feature is in range
            feat_vals = input_features_mean[:, feature_idx]
            selected_sample_indices = np.where((feat_vals >= min_val) & (feat_vals <= max_val))[0]
            update_all_plots(selected_sample_indices)
        return input_selection_fn

    def lime_selection_fn(trace, points, selector):
        # Aggregate selection across all feature traces to handle multi-trace selection
        all_selected_indices = set()
        has_selection = False
        
        # Iterate over 'All' traces (even indices)
        for i in range(0, len(lime_fig.data), 2):
            t = lime_fig.data[i]
            
            # Determine indices for this trace
            if t.uid == trace.uid:
                inds = points.point_inds
            else:
                inds = t.selectedpoints
                
            if inds:
                has_selection = True
                all_selected_indices.update(inds)
        
        if has_selection:
            update_all_plots(list(all_selected_indices))
        else:
            update_all_plots([])

    # Attach Callbacks
    error_fig.data[0].on_selection(error_selection_fn)
    
    for i, fig in enumerate(input_figs):
        fig.data[0].on_selection(input_selection_fn_factory(i))
        
    if lime_fig is not None:
        for i in range(len(lime_df.columns)):
            lime_fig.data[2*i].on_selection(lime_selection_fn)
    
    # Layout
    input_grid = widgets.GridBox(input_figs, layout=widgets.Layout(grid_template_columns="repeat(2, 1fr)"))
    items = [error_fig, input_grid]
    if lime_fig:
        items.append(lime_fig)
        
    display(widgets.VBox(items))


Calculating predictions for all samples...
LIME data available for 4 samples.


    'data': [{'marker': {'color': 'lightgrey'},
              'name': 'All Resid…